# Day 27 - Explain Cache and Performance Debugging

**Topic:** Explain Cache and Performance Debugging  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** ฝึกอ่าน `.explain()` เพื่อเข้าใจ Spark physical plan, ใช้ `cache()` / `persist()` อย่างตั้งใจเพื่อลด repeated computation และฝึก performance debugging mindset เบื้องต้น

วันนี้เราจะฝึก debug performance แบบ practical ใน PySpark โดยเริ่มจากการอ่าน execution plan ด้วย `.explain()`, มองหา operation ที่มักแพง เช่น `Exchange`, `Join`, `Aggregate`, แล้วทดลองใช้ `cache()` / `persist()` เมื่อมี DataFrame ที่ถูก reuse หลายครั้งใน notebook หรือ pipeline step เดียวกัน


## Goal of the day

1. ใช้ `.explain()` เพื่อดู execution plan เบื้องต้นได้
2. แยกให้ออกว่า plan ส่วนไหนคือ `Filter`, `Project`, `Join`, `Aggregate`, `Exchange` หรือ `InMemoryTableScan`
3. เข้าใจว่า `cache()` และ `persist()` เป็น lazy เหมือน transformation และต้องมี action เพื่อให้ Spark คำนวณ DataFrame และเก็บผลลัพธ์ลง cache จริง
4. ใช้ `cache()` / `persist()` เฉพาะกับ intermediate DataFrame ที่จะถูก reuse หลายครั้งจริง ๆ เพื่อหลีกเลี่ยง repeated computation
5. สร้าง performance debugging checklist แบบที่ Data Engineer ใช้งานจริงได้


## Why it matters in production

ใน production pipeline ปัญหา performance ไม่ได้แก้ด้วยการเดาสุ่มว่า “ใส่ cache แล้วจะเร็วขึ้น” เสมอไป แต่ต้อง debug จากหลักฐาน เช่น execution plan, shuffle, join strategy, data size, data skew และ repeated actions

เรื่องนี้สำคัญเพราะ:

- `.explain()` ช่วยให้เห็นว่า Spark วางแผนจะ execute งานอย่างไร เช่น read, filter, join, aggregate และ shuffle
- `Exchange` ใน physical plan เป็นสัญญาณสำคัญว่า Spark มีการ redistribute data ซึ่งมักเกี่ยวข้องกับ shuffle และอาจเป็นจุดแพงของ job
- `cache()` / `persist()` ช่วยลด repeated computation ได้ ถ้า DataFrame เดิมถูกใช้โดยหลาย action หรือหลาย downstream branches
- cache ที่ใช้ผิดที่อาจกิน memory มากเกินไป จน Spark ต้อง spill ข้อมูลลง disk และอาจทำให้ job อื่นช้าลง
- performance debugging ที่ดีต้องดูทั้ง code, physical plan, runtime metrics จาก Spark UI / Fabric monitoring และ validate output ว่าผลลัพธ์ยังถูกต้อง

Production takeaway:

> Performance debugging ที่ดีเริ่มจากการอ่าน plan และหา bottleneck ก่อน แล้วค่อยเลือก optimize pattern ให้ตรงจุด ไม่ใช่ใส่ cache ทุก DataFrame


## Key concepts

| Concept | Meaning |
|---|---|
| `explain()` | แสดง execution plan ของ DataFrame เพื่อดูว่า Spark จะ execute อย่างไร |
| Logical Plan | แผนเชิง logic ว่า query ต้องทำอะไร เช่น filter, select, join, aggregate |
| Physical Plan | แผน execution ที่ Spark optimizer เลือกใช้เพื่อทำงานจริง เช่น join strategy, shuffle, scan, aggregate |
| `Project` | การเลือกหรือสร้าง columns เช่น `select`, `withColumn` |
| `Filter` | การกรอง records เช่น `where`, `filter` |
| `Exchange` | จุดที่ Spark ต้อง redistribute data หรือ shuffle ข้อมูลระหว่าง Spark partitions |
| `Join` | การเชื่อม DataFrame หลายตัว อาจเกิด shuffle หรือ broadcast ได้ตาม strategy |
| `Aggregate` | การ group / summarize data มักเป็น wide transformation และอาจมี shuffle |
| `cache()` | mark DataFrame ให้ Spark เก็บผลลัพธ์ไว้ใน memory เมื่อ DataFrame นั้นถูก materialize ผ่าน action |
| `persist()` | คล้าย cache แต่เลือก `StorageLevel` ได้ เช่น memory, disk, memory and disk |
| Materialization | การ trigger action เช่น `.count()` หรือ `.show()` เพื่อให้ Spark compute DataFrame และเก็บผลลัพธ์ของ cached/persisted DataFrame ไว้จริง |
| `unpersist()` | เอาผลลัพธ์ที่ Spark cache/persist ไว้ออกจาก memory/storage เพื่อคืน resource ให้ Spark cluster |


## Concept explanation

### `.explain()` ใช้ดูอะไร?

เวลาเขียน transformation chain เช่น:

```python
df_result = (
    df_transactions
    .filter(F.col("amount") > 0)
    .join(df_customers, "customer_id", "left")
    .groupBy("region")
    .agg(F.sum("amount").alias("total_amount"))
)
```

Spark ยังไม่ได้ execute จริงจนกว่าจะมี action แต่เราสามารถใช้ `.explain()` เพื่อดูว่า Spark วางแผนไว้อย่างไร

คำที่ควรเริ่มสังเกตใน physical plan:

- `Filter` = มีการกรองข้อมูล
- `Project` = มีการเลือกหรือสร้าง column
- `Exchange` = มี shuffle / repartition เกิดขึ้น
- `SortMergeJoin`, `BroadcastHashJoin` = Spark เลือก join strategy แบบไหน
    - `SortMergeJoin` = ทั้งสอง table ต้อง shuffle และ sort ก่อน join มักเกิดเมื่อทั้งสองฝั่งใหญ่เกินจะ broadcast
    - `BroadcastHashJoin` = table เล็กถูก broadcast ไปทุก executor แล้ว join แบบ local โดยไม่ต้อง shuffle ฝั่งใหญ่
- `HashAggregate` = มี aggregation
- `InMemoryTableScan` = plan กำลังอ่านข้อมูลจาก cached DataFrame

> `.explain()` ไม่ได้แทน Spark UI / Fabric monitoring ทั้งหมด แต่เป็นจุดเริ่มต้นที่ดีมากในการเข้าใจว่า query แพงตรงไหน

### Cache / Persist ใช้ตอนไหน?

ควรคิดถึง cache / persist เมื่อมี DataFrame ที่:

1. สร้างจาก transformation ที่ค่อนข้างแพง เช่น join + aggregation
2. ถูกใช้ซ้ำหลาย action หรือหลาย downstream branches ทั้งใน pipeline และระหว่าง performance investigation
3. มีขนาดพอเหมาะกับ memory และ storage level ที่เลือกใน Spark cluster

ตัวอย่าง mindset:

```python
df_reusable = expensive_transformation.cache()
df_reusable.count()   # materialize cache
df_reusable.show()    # reuse cached result
df_reusable.write.saveAsTable("some_table")
df_reusable.unpersist()
```

### Cache ไม่ใช่ performance magic

Cache อาจช่วยได้ถ้า reuse จริง แต่ cache อาจทำให้แย่ลงถ้า:

- DataFrame ใหญ่มากจน memory ไม่พอ
- ใช้ DataFrame แค่ครั้งเดียวแล้ว cache
- ลืม `unpersist()` หลังเลิกใช้ cached DataFrame ทำให้ memory/storage ถูกถือค้างใน session ที่รันต่อเนื่องนาน
- cache intermediate DataFrame ที่สร้างจาก transformation ธรรมดา ไม่ใช่งานแพงอย่าง join หรือ aggregation
- cache ก่อน filter หรือก่อน select เฉพาะ columns ที่จำเป็น ทำให้ Spark เก็บข้อมูลที่ไม่จำเป็นไว้ใน cache

หลักคิดง่าย ๆ:

> Cache หลังจากลด data size แล้ว และก่อนจุดที่ DataFrame จะถูก reuse หลายครั้ง


## Mermaid diagram: Performance Debugging Flow

```mermaid
flowchart LR
    A[PySpark transformation chain] --> B["Run .explain(mode='formatted')"]
    B --> C{See expensive pattern?}
    C -- Exchange / Join / Aggregate --> D[Check shuffle, join strategy, data distribution]
    C -- Repeated actions --> E[Consider cache or persist]
    E --> F[Materialize with action: count/show/write]
    F --> G[Reuse cached DataFrame]
    G --> H[Validate output]
    H --> I[unpersist or clear cache]
    D --> J[Check Spark UI or Fabric monitoring]
    J --> H
```

Key idea:

> Debug ก่อน optimize: ดู physical plan ก่อนว่า Spark execute งานอย่างไร, ตรวจว่ามี action ไหนทำให้ DataFrame ถูก compute ซ้ำ แล้วค่อยใช้ `cache()` / `persist()` อย่างมีเหตุผล


## Setup / imports


In [1]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark import StorageLevel  # Used to choose storage level for persist(), such as MEMORY_AND_DISK.

# Small shuffle partition setting for this lab only.
# In production, tune this based on data size and cluster capacity.
spark.conf.set("spark.sql.shuffle.partitions", "4")

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 3, Finished, Available, Finished, False)

## Create mock data


In [2]:
transactions_data = [
    ("T001", 101, "P01", "2026-01-01", 1200.00, "success", "web", "2026-01-01 09:10:00"),
    ("T002", 102, "P01", "2026-01-01", 450.00, "success", "mobile", "2026-01-01 09:20:00"),
    ("T003", 103, "P02", "2026-01-02", 0.00, "failed", "web", "2026-01-02 10:05:00"),
    ("T004", 104, "P02", "2026-01-02", 2200.00, "success", "branch", "2026-01-02 11:30:00"),
    ("T005", 105, "P03", "2026-01-03", 780.00, "success", "mobile", "2026-01-03 12:00:00"),
    ("T006", 101, "P01", "2026-01-03", 1500.00, "success", "web", "2026-01-03 13:15:00"),
    ("T007", 102, "P03", "2026-01-04", -50.00, "success", "mobile", "2026-01-04 14:10:00"),
    ("T008", 106, "P04", "2026-01-04", 300.00, "pending", "partner", "2026-01-04 15:00:00"),
    ("T009", 103, "P02", "2026-01-05", 640.00, "success", "web", "2026-01-05 16:25:00"),
    ("T010", 104, "P02", "2026-01-05", 1800.00, "success", "branch", "2026-01-05 17:05:00"),
    ("T011", 101, "P03", "2026-01-06", 950.00, "success", "mobile", "2026-01-06 10:40:00"),
    ("T012", 105, "P04", "2026-01-06", 1250.00, "success", "partner", "2026-01-06 12:55:00"),
]

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("property_id", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("payment_status", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
    T.StructField("updated_at", T.StringType(), True),
])

df_transactions_raw = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions = (
    df_transactions_raw
    .withColumn("transaction_date", F.to_date("transaction_date"))
    .withColumn("updated_at", F.to_timestamp("updated_at"))
)

customers_data = [
    (101, "Alice", "premium", "Bangkok", "active"),
    (102, "Bob", "standard", "Bangkok", "active"),
    (103, "Charlie", "standard", "Chiang Mai", "active"),
    (104, "Diana", "premium", "Phuket", "active"),
    (105, "Ethan", "standard", "Rayong", "inactive"),
]

customer_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("customer_name", T.StringType(), True),
    T.StructField("segment", T.StringType(), True),
    T.StructField("region", T.StringType(), True),
    T.StructField("customer_status", T.StringType(), True),
])

df_customers = spark.createDataFrame(customers_data, customer_schema)

df_transactions.show(truncate=False)
df_customers.show(truncate=False)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 4, Finished, Available, Finished, False)

+--------------+-----------+-----------+----------------+------+--------------+-------------+-------------------+
|transaction_id|customer_id|property_id|transaction_date|amount|payment_status|source_system|updated_at         |
+--------------+-----------+-----------+----------------+------+--------------+-------------+-------------------+
|T001          |101        |P01        |2026-01-01      |1200.0|success       |web          |2026-01-01 09:10:00|
|T002          |102        |P01        |2026-01-01      |450.0 |success       |mobile       |2026-01-01 09:20:00|
|T003          |103        |P02        |2026-01-02      |0.0   |failed        |web          |2026-01-02 10:05:00|
|T004          |104        |P02        |2026-01-02      |2200.0|success       |branch       |2026-01-02 11:30:00|
|T005          |105        |P03        |2026-01-03      |780.0 |success       |mobile       |2026-01-03 12:00:00|
|T006          |101        |P01        |2026-01-03      |1500.0|success       |web      

## PySpark code examples

ส่วนนี้จะไล่จากการสร้าง transformation chain, อ่าน physical plan, trigger action, ทดลอง repeated actions, ใช้ cache / persist และเขียนผลลัพธ์ลง Lakehouse table แบบเล็ก ๆ


### Example 1: Build a transformation chain without running an action

Cell นี้ยังไม่มี `.show()`, `.count()` หรือ `.write` ดังนั้น Spark จะยังไม่ execute จริง แค่สร้าง logical plan ไว้ก่อน


In [3]:
df_success_transactions = (
    df_transactions
    .filter((F.col("payment_status") == "success") & (F.col("amount") > 0))
    .select(
        "transaction_id",
        "customer_id",
        "property_id",
        "transaction_date",
        "amount",
        "source_system",
    )
    .withColumn(
        "amount_bucket",
        F.when(F.col("amount") >= 1000, F.lit("high")).otherwise(F.lit("normal")),
    )
)

print("Transformation chain created. No action has been triggered in this cell.")

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 5, Finished, Available, Finished, False)

Transformation chain created. No action has been triggered in this cell.


### Example 2: Use explain to inspect the physical plan

ให้ดูว่าใน physical plan มี `Filter` และ `Project` หรือไม่


In [4]:
df_success_transactions.explain(mode="formatted")

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 6, Finished, Available, Finished, False)

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [8]: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733]
Arguments: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [8]: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733]
Condition : ((isnotnull(payment_status#731) AND isnotnull(amount#730)) AND ((payment_status#731 = success) AND (amount#730 > 0.0)))

(3) Project [codegen id : 1]
Output [7]: [transaction_id#726, customer_id#727, property_id#728, cast(transaction_date#729 as date) AS

### Example 3: Add join and aggregation, then inspect physical plan

Join + aggregation เป็น pattern ที่มักแพงกว่า filter/select เพราะอาจเกิด shuffle ให้สังเกตคำอย่าง `Exchange`, `Join`, `HashAggregate` หรือ join strategy ที่ Spark เลือกใน physical plan


In [5]:
df_customer_revenue = (
    df_success_transactions
    .join(df_customers, on="customer_id", how="left")
    .groupBy("region", "segment")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.round(F.avg("amount"), 2).alias("avg_amount"),
    )
    .orderBy(F.desc("total_amount"))
)

df_customer_revenue.explain(mode="formatted")

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 7, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Sort (16)
   +- Exchange (15)
      +- HashAggregate (14)
         +- Exchange (13)
            +- HashAggregate (12)
               +- Project (11)
                  +- SortMergeJoin LeftOuter (10)
                     :- Sort (5)
                     :  +- Exchange (4)
                     :     +- Project (3)
                     :        +- Filter (2)
                     :           +- Scan ExistingRDD (1)
                     +- Sort (9)
                        +- Exchange (8)
                           +- Project (7)
                              +- Scan ExistingRDD (6)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733]
Arguments: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733], MapPartitionsRDD[44] at applySchema

Trigger action ด้วย `.show()` เพื่อดู result จริง


In [6]:
df_customer_revenue.show(truncate=False)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 8, Finished, Available, Finished, False)

+----------+--------+-----------------+------------+----------+
|region    |segment |transaction_count|total_amount|avg_amount|
+----------+--------+-----------------+------------+----------+
|Phuket    |premium |2                |4000.0      |2000.0    |
|Bangkok   |premium |3                |3650.0      |1216.67   |
|Rayong    |standard|2                |2030.0      |1015.0    |
|Chiang Mai|standard|1                |640.0       |640.0     |
|Bangkok   |standard|1                |450.0       |450.0     |
+----------+--------+-----------------+------------+----------+



### Example 4: Repeated actions can recompute the same physical plan

ถ้า DataFrame ยังไม่ได้ cache การเรียกหลาย action เช่น `.show()` แล้ว `.count()` จะทำให้ Spark compute physical plan เดิมซ้ำทุกครั้ง ซึ่งแพงมากถ้า plan มี join หรือ aggregation


In [7]:
df_high_revenue_segments = df_customer_revenue.filter(F.col("total_amount") >= 1000)

df_high_revenue_segments.show(truncate=False)
print("High revenue segment count:", df_high_revenue_segments.count())

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 9, Finished, Available, Finished, False)

+-------+--------+-----------------+------------+----------+
|region |segment |transaction_count|total_amount|avg_amount|
+-------+--------+-----------------+------------+----------+
|Phuket |premium |2                |4000.0      |2000.0    |
|Bangkok|premium |3                |3650.0      |1216.67   |
|Rayong |standard|2                |2030.0      |1015.0    |
+-------+--------+-----------------+------------+----------+

High revenue segment count: 3


### Example 5: Cache a reusable DataFrame intentionally

`cache()` เป็น lazy เช่นกัน หมายความว่า Spark จะ mark DataFrame ไว้ก่อน แต่จะ materialize จริงเมื่อมี action เช่น `.count()`


In [8]:
df_cached_customer_revenue = df_customer_revenue.cache()

print("Is cached after calling cache():", df_cached_customer_revenue.is_cached)
print("Storage level:", df_cached_customer_revenue.storageLevel)

# Materialize the cache with an action.
cached_count = df_cached_customer_revenue.count()
print("Cached row count:", cached_count)

# This action can reuse the cached result.
df_cached_customer_revenue.show(truncate=False)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 10, Finished, Available, Finished, False)

Is cached after calling cache(): True
Storage level: Disk Memory Deserialized 1x Replicated
Cached row count: 5
+----------+--------+-----------------+------------+----------+
|region    |segment |transaction_count|total_amount|avg_amount|
+----------+--------+-----------------+------------+----------+
|Phuket    |premium |2                |4000.0      |2000.0    |
|Bangkok   |premium |3                |3650.0      |1216.67   |
|Rayong    |standard|2                |2030.0      |1015.0    |
|Chiang Mai|standard|1                |640.0       |640.0     |
|Bangkok   |standard|1                |450.0       |450.0     |
+----------+--------+-----------------+------------+----------+



หลังจาก cache ถูก materialize แล้ว ควรตรวจสอบจากหลายแหล่งประกอบกันเพื่อยืนยันว่า cache สำเร็จจริง เช่น `is_cached`, `storageLevel`, Spark UI / Fabric monitoring และ physical plan

ในบาง runtime หรือบาง plan อาจเห็น `InMemoryTableScan` / `InMemoryRelation` ใน physical plan ซึ่งเป็นสัญญาณว่า Spark อ่านจาก cached result

> หมายเหตุ: ถ้า `.explain()` ไม่แสดง `InMemoryTableScan` ไม่ได้แปลว่า cache ไม่สำเร็จเสมอไป ควรดูจากหลักฐานอื่นประกอบด้วย

In [10]:
df_cached_customer_revenue.explain(mode="formatted")

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 12, Finished, Available, Finished, False)

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Sort (16)
   +- Exchange (15)
      +- HashAggregate (14)
         +- Exchange (13)
            +- HashAggregate (12)
               +- Project (11)
                  +- SortMergeJoin LeftOuter (10)
                     :- Sort (5)
                     :  +- Exchange (4)
                     :     +- Project (3)
                     :        +- Filter (2)
                     :           +- Scan ExistingRDD (1)
                     +- Sort (9)
                        +- Exchange (8)
                           +- Project (7)
                              +- Scan ExistingRDD (6)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733]
Arguments: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733], MapPartitionsRDD[44] at applySchema

### Example 6: Use persist when you want to choose storage level

`persist()` คล้าย `cache()` แต่สามารถเลือก storage level ได้ เช่น `MEMORY_AND_DISK` ซึ่งช่วยให้ Spark เก็บข้อมูลไว้ใน memory ก่อน และใช้ disk ร่วมด้วยเมื่อ memory ไม่พอ


In [11]:
df_persisted_success_transactions = df_success_transactions.persist(StorageLevel.MEMORY_AND_DISK)

# Materialize persisted DataFrame.
persisted_count = df_persisted_success_transactions.count()

print("Persisted row count:", persisted_count)
print("Persisted storage level:", df_persisted_success_transactions.storageLevel)

df_persisted_success_transactions.groupBy("source_system").agg(
    F.count("*").alias("success_transaction_count"),
    F.round(F.sum("amount"), 2).alias("success_amount"),
).orderBy("source_system").show(truncate=False)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 13, Finished, Available, Finished, False)

Persisted row count: 9
Persisted storage level: Disk Memory Serialized 1x Replicated
+-------------+-------------------------+--------------+
|source_system|success_transaction_count|success_amount|
+-------------+-------------------------+--------------+
|branch       |2                        |4000.0        |
|mobile       |3                        |2180.0        |
|partner      |1                        |1250.0        |
|web          |3                        |3340.0        |
+-------------+-------------------------+--------------+



หลังใช้ cache / persist เสร็จ ควร `unpersist()` เมื่อไม่ต้อง reuse แล้ว เพื่อคืน resource ให้ Spark session


In [12]:
df_persisted_success_transactions.unpersist()
df_cached_customer_revenue.unpersist()

print("Unpersisted cached DataFrames used in examples.")

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 14, Finished, Available, Finished, False)

Unpersisted cached DataFrames used in examples.


### Example 7: Write the debug result to a Fabric Lakehouse table

In [13]:
table_name = "day27_customer_revenue_debug"

(
    df_customer_revenue
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

spark.read.table(table_name).show(truncate=False)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 15, Finished, Available, Finished, False)

+----------+--------+-----------------+------------+----------+
|region    |segment |transaction_count|total_amount|avg_amount|
+----------+--------+-----------------+------------+----------+
|Phuket    |premium |2                |4000.0      |2000.0    |
|Bangkok   |premium |3                |3650.0      |1216.67   |
|Rayong    |standard|2                |2030.0      |1015.0    |
|Chiang Mai|standard|1                |640.0       |640.0     |
|Bangkok   |standard|1                |450.0       |450.0     |
+----------+--------+-----------------+------------+----------+



### Example 8: Create a lightweight performance debugging checklist

ในงานจริง checklist ไม่ได้แทน monitoring แต่ช่วยให้เรา debug อย่างเป็นขั้นตอนแทนการเดาสุ่ม


In [20]:
debug_checklist_data = [
    (1, "Symptom", "Job is slow or repeated actions take long", "Check Spark UI / Fabric monitoring and notebook actions"),
    (2, "Plan", "Physical plan has Exchange", "Investigate shuffle, join key, aggregation key, partitioning"),
    (3, "Join", "Join stage is expensive", "Check join strategy, broadcast opportunity, skewed keys"),
    (4, "Reuse", "Same expensive DataFrame is used multiple times", "Consider cache or persist after filter and column selection"),
    (5, "Memory", "Cache makes job slower or causes memory pressure", "Unpersist unused DataFrames and avoid caching huge intermediate data"),
    (6, "Output", "Result looks wrong after optimization", "Validate row counts, aggregated totals, and join completeness after optimization"),
]

debug_checklist_schema = [
    "step_order",
    "debug_area",
    "signal",
    "suggested_check",
]

df_debug_checklist = spark.createDataFrame(debug_checklist_data, debug_checklist_schema)

df_debug_checklist.show(truncate=False)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 22, Finished, Available, Finished, False)

+----------+----------+------------------------------------------------+--------------------------------------------------------------------------------+
|step_order|debug_area|signal                                          |suggested_check                                                                 |
+----------+----------+------------------------------------------------+--------------------------------------------------------------------------------+
|1         |Symptom   |Job is slow or repeated actions take long       |Check Spark UI / Fabric monitoring and notebook actions                         |
|2         |Plan      |Physical plan has Exchange                      |Investigate shuffle, join key, aggregation key, partitioning                    |
|3         |Join      |Join stage is expensive                         |Check join strategy, broadcast opportunity, skewed keys                         |
|4         |Reuse     |Same expensive DataFrame is used multiple times |Cons

## Quick recap

| Question | Answer |
|---|---|
| `.explain(mode="formatted")` ใช้ทำอะไร? | ใช้ดู physical plan เพื่อเข้าใจว่า Spark จะ execute งานอย่างไร |
| `.explain()` เป็น action ไหม? | โดยทั่วไปไม่ใช่ action ที่ compute data result แต่เป็นการแสดง plan |
| เห็น `Exchange` แปลว่าอะไร? | มักเกี่ยวข้องกับ shuffle / data redistribution ระหว่าง Spark partitions |
| `cache()` ทำงานทันทีไหม? | ไม่ทันที ต้องมี action เพื่อ materialize cache |
| ควร cache ตอนไหน? | ตอนที่ DataFrame สร้างจาก transformation แพงและจะถูก reuse หลายครั้ง โดยควร filter และ select columns ที่จำเป็นก่อน cache |
| ทำไมต้อง `unpersist()`? | เพื่อคืน memory / storage ให้ Spark session เมื่อไม่ต้องใช้ cached data แล้ว |


## Exercises


### Exercise 1: Inspect physical plan for a filtered transaction DataFrame

สร้าง DataFrame ชื่อ `df_branch_or_web_success` สำหรับ transaction ที่:

Requirements:

- `payment_status = "success"`
- `source_system` เป็น `web` หรือ `branch`
- `amount > 500`
- select เฉพาะ `transaction_id`, `customer_id`, `transaction_date`, `amount`, `source_system`
- เพิ่ม column `debug_note = "filtered_for_physical_plan_review"`
- ใช้ `.explain(mode="formatted")` ก่อน `.show()`

Expected result:

- เห็น physical plan ที่มี filter / project
- เห็นเฉพาะ success transactions จาก web หรือ branch ที่ amount มากกว่า 500


In [21]:
df_branch_or_web_success = (
    df_transactions
    .filter(
        (F.col("payment_status") == "success")
        & (F.col("source_system").isin("web", "branch"))
        & (F.col("amount") > 500)
    )
    .select(
        "transaction_id",
        "customer_id",
        "transaction_date",
        "amount",
        "source_system",
    )
    .withColumn("debug_note", F.lit("filtered_for_physical_plan_review"))
)

df_branch_or_web_success.explain(mode="formatted")
df_branch_or_web_success.show(truncate=False)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 23, Finished, Available, Finished, False)

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [8]: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733]
Arguments: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [8]: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733]
Condition : ((isnotnull(payment_status#731) AND isnotnull(amount#730)) AND (((payment_status#731 = success) AND source_system#732 IN (web,branch)) AND (amount#730 > 500.0)))

(3) Project [codegen id : 1]
Output [6]: [transaction_id#726, customer_id#727, cast(transac

### Exercise 2: Persist a reusable summary and clean it up

สร้าง summary ชื่อ `df_region_source_summary` จาก `df_success_transactions` โดย join กับ `df_customers` แล้วสรุปตาม `region` และ `source_system`

Requirements:

- join กับ `df_customers` ด้วย `customer_id`
- group by `region`, `source_system`
- aggregate `transaction_count`, `total_amount`
- ใช้ `persist(StorageLevel.MEMORY_AND_DISK)`
- materialize cache ด้วย `.count()`
- แสดงผลด้วย `.show()`
- print storage level
- ปิดท้ายด้วย `.unpersist()`

Expected result:

- ได้ summary ราย region/source system
- เห็นว่า persisted DataFrame มี storage level
- cleanup cache หลังใช้เสร็จ


In [22]:
df_region_source_summary = (
    df_success_transactions
    .join(df_customers, on="customer_id", how="left")
    .groupBy("region", "source_system")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
    )
    .orderBy("region", "source_system")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

summary_count = df_region_source_summary.count()
print("Summary row count:", summary_count)
print("Storage level:", df_region_source_summary.storageLevel)

df_region_source_summary.show(truncate=False)

df_region_source_summary.unpersist()

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 24, Finished, Available, Finished, False)

Summary row count: 6
Storage level: Disk Memory Serialized 1x Replicated
+----------+-------------+-----------------+------------+
|region    |source_system|transaction_count|total_amount|
+----------+-------------+-----------------+------------+
|Bangkok   |mobile       |2                |1400.0      |
|Bangkok   |web          |2                |2700.0      |
|Chiang Mai|web          |1                |640.0       |
|Phuket    |branch       |2                |4000.0      |
|Rayong    |mobile       |1                |780.0       |
|Rayong    |partner      |1                |1250.0      |
+----------+-------------+-----------------+------------+



DataFrame[region: string, source_system: string, transaction_count: bigint, total_amount: double]

### Exercise 3: Compare normal join plan and broadcast join plan

ใช้ `.explain()` เพื่อ compare plan ระหว่าง normal join และ broadcast join แบบง่าย ๆ

Requirements:

- สร้าง `df_normal_join_debug` โดย join `df_transactions` กับ `df_customers` แบบ left join
- สร้าง `df_broadcast_join_debug` โดยใช้ `F.broadcast(df_customers)`
- call `.explain(mode="formatted")` ทั้งสอง DataFrame
- call `.count()` ทั้งสอง DataFrame เพื่อ trigger action

Expected result:

- normal join และ broadcast join มี physical plan ต่างกัน
- broadcast plan จะเห็นคำว่า `BroadcastExchange` หรือ `BroadcastHashJoin`
- result count ของทั้งสอง DataFrame ควรเท่ากัน


In [23]:
df_normal_join_debug = df_transactions.join(
    df_customers,
    on="customer_id",
    how="left",
)

df_broadcast_join_debug = df_transactions.join(
    F.broadcast(df_customers),
    on="customer_id",
    how="left",
)

print("Normal join plan")
df_normal_join_debug.explain(mode="formatted")

print("Broadcast join plan")
df_broadcast_join_debug.explain(mode="formatted")

normal_join_count = df_normal_join_debug.count()
broadcast_join_count = df_broadcast_join_debug.count()

print("Normal join count:", normal_join_count)
print("Broadcast join count:", broadcast_join_count)
print("Counts match:", normal_join_count == broadcast_join_count)

StatementMeta(, f1ac1085-3bbe-4dd4-999e-feb16b72dfb3, 25, Finished, Available, Finished, False)

Normal join plan
== Physical Plan ==
AdaptiveSparkPlan (10)
+- Project (9)
   +- SortMergeJoin LeftOuter (8)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Project (2)
      :        +- Scan ExistingRDD (1)
      +- Sort (7)
         +- Exchange (6)
            +- Scan ExistingRDD (5)


(1) Scan ExistingRDD
Output [8]: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733]
Arguments: [transaction_id#726, customer_id#727, property_id#728, transaction_date#729, amount#730, payment_status#731, source_system#732, updated_at#733], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project
Output [8]: [transaction_id#726, customer_id#727, property_id#728, cast(transaction_date#729 as date) AS transaction_date#742, amount#730, payment_status#731, source_system#732, cast(updated_at#733 as timestamp) AS updated_at#751]
Input

## Common mistakes

1. **คิดว่า `.explain()` คือการ run job จริง**

   `.explain()` ใช้ดู physical plan ไม่ใช่ action ที่ compute result แบบ `.show()`, `.count()` หรือ `.write` แต่ physical plan ที่เห็นอาจช่วยบอกได้ว่า action ถัดไปน่าจะทำงานอย่างไร

2. **เรียก `cache()` แล้วคิดว่าข้อมูลถูกเก็บทันที**

   `cache()` แค่ mark DataFrame ไว้ ต้องมี action เช่น `.count()` เพื่อ materialize cache จริง

3. **cache ทุก DataFrame เพื่อหวังให้เร็วขึ้น**

   Cache มีต้นทุนด้าน memory/storage ถ้าใช้กับ DataFrame ใหญ่หรือใช้ครั้งเดียว อาจทำให้ job ช้าลงแทนที่จะเร็วขึ้น

4. **ลืม `unpersist()` หลัง debug เสร็จ**

   Notebook session ที่รันนานอาจสะสม cached DataFrames ไว้โดยไม่จำเป็น ทำให้ memory เหลือน้อยลงและกระทบ job ถัดไป

5. **ดูแค่ duration แต่ไม่ดู physical plan**

   Job ที่ช้าอาจมาจาก shuffle, data skew, join strategy, small files, repeated action หรือ cluster resource ต้องดูทั้ง physical plan และ runtime evidence ประกอบกัน

6. **ใช้ `collect()` เพื่อ debug ข้อมูลใหญ่**

   `collect()` ดึงทุก row ของ DataFrame กลับมาไว้ที่ driver memory ทั้งหมด และเสี่ยงที่ driver จะเกิด OOM ดังนั้น ควรใช้ `show()`, `limit()`, `sample()` หรือ aggregate summary ก่อนเสมอ

7. **เชื่อว่า physical plan จะเหมือนกันทุก runtime เสมอ**

   Spark runtime, configuration, data size และ Adaptive Query Execution อาจทำให้ physical plan เปลี่ยนได้ โดยเฉพาะตอนรันจริง ดังนั้นควรใช้ `.explain()` คู่กับ Spark UI / Fabric monitoring เมื่อต้อง debug performance ของงานจริง


## Expected Output / Success Criteria

เมื่อจบ Day 27 ควรทำได้:

- ใช้ `.explain(mode="formatted")` เพื่ออ่าน Spark execution plan เบื้องต้นได้
- มองหา keyword สำคัญใน plan เช่น `Filter`, `Project`, `Exchange`, `Join`, `HashAggregate`, `InMemoryTableScan`
- อธิบายได้ว่า `Exchange` มักเกี่ยวกับ shuffle และควรถูกตรวจเพิ่มเมื่อ job ช้า
- อธิบายได้ว่า `cache()` / `persist()` เป็น lazy และต้องใช้ action เพื่อ materialize cache
- ใช้ cache / persist อย่างมีเหตุผลเมื่อมี repeated computation จริง
- cleanup cached DataFrame ด้วย `unpersist()` เมื่อไม่ใช้แล้ว
- เขียน performance debugging checklist แบบ practical สำหรับ Spark job ได้


## Final takeaway

Performance debugging ใน Spark ไม่ใช่การหา single magic command แต่คือการอ่านหลักฐานให้เป็น แล้ว optimize ให้ตรงจุด

> ดู physical plan ก่อน เดาให้น้อยลง ใช้ `cache()` / `persist()` เฉพาะเมื่อ reuse จริง และ validate result ทุกครั้งหลังปรับ performance

สิ่งที่ต้องจำสำหรับ Day 27:

- `.explain()` ช่วยให้เห็นว่า Spark จะ scan, filter, join, aggregate และ shuffle อย่างไร
- `cache()` / `persist()` ช่วยลด repeated computation ได้ แต่ต้อง materialize cache ด้วย action
- Cache ที่ดีควรอยู่หลังการลด data size และก่อนจุดที่ DataFrame จะถูก reuse หลายครั้ง
- `Exchange` เป็นสัญญาณสำคัญว่าควรตรวจ shuffle, join key, aggregation key หรือ data distribution ต่อ
- หลัง debug เสร็จควร `unpersist()` เพื่อคืน resource ให้ Spark session


## Optional cleanup


In [ ]:
# spark.sql("DROP TABLE IF EXISTS day27_customer_revenue_debug")
# spark.catalog.clearCache()